In [1]:
INPUT_CSV = "dataset/delivery_orders_march.csv"
OUTPUT_CSV = "result/result.csv"

# Import Data

In [2]:
import pandas

raw_data = pandas.read_csv(INPUT_CSV)

In [3]:
from datetime import datetime, timedelta, tzinfo


class UTC0800(tzinfo):
    _offset = timedelta(seconds = 28800)
    _dst = timedelta(0)
    _name = "UTC+0800"
    def utcoffset(self, dt):
        return self.__class__._offset
    def dst(self, dt):
        return self.__class__._dst
    def tzname(self, dt):
        return self.__class__._name


LOCAL = UTC0800()
    

def to_unix(x):
    return int((x - datetime(1970,1,1)).total_seconds())


def normalize_time(s):
    _temp = datetime.fromtimestamp(s, tz=LOCAL)
    _temp = _temp.replace(hour=0, minute=0, second=0)
    return int((_temp - datetime(1970,1,1, tzinfo=LOCAL)).total_seconds())


def encode_address(s):
    if "metro manila" in s[-12:].lower():
        return 0
    elif "luzon" in s[-5:].lower():
        return 1
    elif "visayas" in s[-7:].lower():
        return 2
    elif "mindanao" in s[-8:].lower():
        return 3
    else:
        return -1

In [4]:
clean_data = raw_data.copy(deep = True)

clean_data["buyeraddress"] = clean_data.buyeraddress.apply(encode_address)
clean_data["selleraddress"] = clean_data.selleraddress.apply(encode_address)

clean_data["2nd_deliver_attempt"] = clean_data["2nd_deliver_attempt"].fillna(clean_data["1st_deliver_attempt"])

clean_data["pick"] = clean_data.pick.apply(normalize_time)
clean_data["1st_deliver_attempt"] = clean_data["1st_deliver_attempt"].apply(normalize_time)
clean_data["2nd_deliver_attempt"] = clean_data["2nd_deliver_attempt"].apply(normalize_time)

# Define Holiday

In [5]:
HOLIDAYS = [
    datetime.strptime("2020-03-08 00:00:00", "%Y-%m-%d %H:%M:%S"),
    datetime.strptime("2020-03-25 00:00:00", "%Y-%m-%d %H:%M:%S"),
    datetime.strptime("2020-03-30 00:00:00", "%Y-%m-%d %H:%M:%S"),
    datetime.strptime("2020-03-31 00:00:00", "%Y-%m-%d %H:%M:%S"),
]

In [6]:
def add_days(x, add):
    for i in range(add):
        x = x + timedelta(days=1)

        while x in HOLIDAYS or x.weekday() == 6:
            x = x + timedelta(days=1)
    
    return x

# First Attempt

In [7]:
SLA = [
    [3, 5, 7, 7],
    [5, 5, 7, 7],
    [7, 7, 7, 7],
    [7, 7, 7, 7],
]

In [8]:
def get_first_attempt(x):
    pick = datetime.fromtimestamp(x["pick"])
    buyer = x["buyeraddress"]
    seller = x["selleraddress"]
    return to_unix(add_days(pick, SLA[buyer][seller]))


clean_data["first"] = clean_data.apply(get_first_attempt, axis=1)

# Second Attempt

In [9]:
def get_second_attempt(x):
    first = datetime.fromtimestamp(x["1st_deliver_attempt"])
    return to_unix(add_days(first, 3))


clean_data["second"] = clean_data.apply(get_second_attempt, axis=1)

# Process

In [10]:
def is_late(x):
    if x["1st_deliver_attempt"] > x["first"]:
        return 1
    elif x["2nd_deliver_attempt"] > x["second"]:
        return 1
    else:
        return 0 


clean_data["is_late"] = clean_data.apply(is_late, axis=1)

# Save Result

In [11]:
result = clean_data.copy(deep = True)


result = result.drop(
    columns=[
        "pick",
        "1st_deliver_attempt",
        "2nd_deliver_attempt",
        "buyeraddress",
        "selleraddress",
        "first",
        "second",
    ]
)

In [12]:
result.to_csv(OUTPUT_CSV, index=False)